# Time series for website
***

This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and exports a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots.reservoirs import plot_reservoir_timeseries, create_reservoir_html

## Configuration

In [2]:
cfg = Config('config_BEAVERS_v100.yml')

# use this meteo dataset
meteo_ds = 'ROCIO-IBEB' # EMO1

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_web = Path('../../docs')
path_layers = path_web / 'layers'
path_ts = path_web / 'timeseries' / 'reservoirs'
path_plots = path_ts / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# point layer
filename = 'reservoirs.geojson'

# decimals in output timeseries
rounding = {
    'storage_mcm': 3,
    'filling': 3,
    'inflow_cms': 3,
    'inflow_mm': 1,
    'outflow_cms': 3,
    'outflow_mm': 1,
    'level_masl': 3,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'storage': 'storage_mcm',
    'inflow': 'inflow_cms',
    'outflow': 'outflow_cms',
    'level': 'level_masl',
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}


## Create time series


In [3]:
# load reservoirs
reservoirs = gpd.read_file(cfg.path_gis / filename).set_index('id')

# process timeseries for each station
for ID in tqdm(reservoirs.index, desc='reservoirs'):

    # reservoir operations
    try:
        resops = pd.read_parquet(path_in / 'resops' / f'{ID}.parquet')
        resops.rename(columns=variables, inplace=True, errors='ignore')
        # compute reservoir filling
        resops['filling'] = resops['storage_mcm'] / reservoirs.loc[ID, 'cap_mcm']
        # compute specific discharge (mm/day)
        for flow in ['inflow', 'outflow']:
            var = f'{flow}_cms'
            if var in resops.columns:
                resops[f'{flow}_mm'] = resops[var] / reservoirs.loc[ID, 'catch_skm'] * 86400 / 1000
        # add degree of regulation to attributes
        if 'inflow_mm' in resops.columns:
            reservoirs.loc[ID, 'dor_d'] = reservoirs.loc[ID, 'cap_mcm'] / (resops['inflow_mm'].mean(skipna=True) * reservoirs.loc[ID, 'catch_skm']) * 1e3
        else:
            reservoirs.loc[ID, 'dor_d'] = pd.NA
    except Exception as e:
        print(f'Error loading discharge timeseries for station {ID}: {e}')
        continue

    # meteo timeseries
    try:
        try:
            meteo = pd.read_parquet(path_in / 'meteo' / meteo_ds / f'{ID}.parquet').loc[ID]
        except:
            meteo = pd.read_parquet(path_in / 'meteo' / 'EMO1' / f'{ID}.parquet').loc[ID]
            meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.rename(columns=variables, inplace=True, errors='ignore')
        # correct dates
        if meteo_ds == 'EMO1':
            meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID}: {e}')
        continue

    # merge timeseries
    start = max(meteo.first_valid_index(), resops.first_valid_index())
    end = min(meteo.last_valid_index(), resops.last_valid_index())
    ts = pd.concat([resops, meteo.loc[start:end]], axis=1)
    ts = ts[ts.columns.intersection(rounding)].round(rounding)
    
    # export timeseries
    ts.to_parquet(path_ts / f'{ID}.parquet')

    # extract attributes
    attrs = reservoirs.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title() if pd.notna(attrs['name']) else '', 
            attrs['river'].title() if pd.notna(attrs['river']) else '', 
            attrs['basin'].title()
        )
            
        fig = plot_reservoir_timeseries(
            ts,
            attrs,
            title=title,
            save=True
        )

        # save plot as HTML
        create_reservoir_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except:
        print(f"The plot for time series {ID} couldn't be created")

# export updated points layer
reservoirs['dor_d'] = reservoirs['dor_d'].astype(float).round(0)
reservoirs.to_file(path_layers / filename)

reservoirs:   0%|          | 0/374 [00:00<?, ?it/s]